In [1]:
import pandas as pd
import time

In [2]:
# load filtered policy data
df_policy = pd.read_csv('../data/clean/eu_capitals_covid_data.csv')

In [3]:
periods = [
    '2020Q1', '2020Q2', '2020Q3', '2020Q4',
    '2021Q1', '2021Q2', '2021Q3', '2021Q4',
    '2022Q1', '2022Q2', '2022Q3', '2022Q4',
]

base_url = "https://aqicn.org/data-platform/covid19/report/50428-9b6996dd/{}"

# Map AQICN English names to your COVID Policy  dataset names
city_mapping = {
    'Vienna': 'Wien', 'Brussels': 'Bruxelles', 'Prague': 'Praha',
    'Copenhagen': 'København', 'Paris': 'Paris', 'Berlin': 'Berlin',
    'Rome': 'Roma', 'Madrid': 'Madrid', 'Stockholm': 'Stockholm',
    'Dublin': 'Dublin', 'Lisbon': 'Lisboa', 'Zagreb': 'Grad Zagreb',
    'Riga': 'Riga', 'Amsterdam': 'Amsterdam', 'Vilnius': 'Vilniaus m.'
}

# Download and filter the AQICN datasets
all_pollution_data = []

print("Downloading AQICN historical CSVs...")
for period in periods:
    print(f"Fetching {period}...")
    url = base_url.format(period)
    
    try:
        # AQICN CSVs usually have 4 lines of text/comments before the actual headers
        df_temp = pd.read_csv(url, skiprows=4)
        
        # Immediately filter to save RAM: Keep only our 15 target cities
        df_temp = df_temp[df_temp['City'].isin(city_mapping.keys())]
        
        if not df_temp.empty:
            all_pollution_data.append(df_temp)
            
        time.sleep(1)  # Brief pause so we don't hammer the AQICN server
        
    except Exception as e:
        print(f"Could not read {period}: {e}")

Fetching 2020Q1...
Fetching 2020Q2...
Fetching 2020Q3...
Fetching 2020Q4...
Fetching 2021Q1...
Fetching 2021Q2...
Fetching 2021Q3...
Fetching 2021Q4...
Fetching 2022Q1...
Fetching 2022Q2...
Fetching 2022Q3...
Fetching 2022Q4...


In [4]:
# Combine and Clean the AQICN Data
df_aqicn = pd.concat(all_pollution_data, ignore_index=True)

# make the variable names unanimous across different data batches
df_aqicn['Specie'] = df_aqicn['Specie'].str.replace('-', ' ', regex=False).str.strip()

# AQICN gives min, max, and median. We will extract the daily 'median'.
# Pivot from "Long" format to "Wide" format
print("Pivoting pollution data...")
df_pivot = df_aqicn.pivot_table(
    index=['City', 'Date'], 
    columns='Specie', 
    values='median'
).reset_index()

# Rename the English city names to match your policy data
df_pivot['city_name'] = df_pivot['City'].map(city_mapping)
df_pivot = df_pivot.drop(columns=['City'])

# Format dates to ensure a perfect match
df_pivot['Date'] = pd.to_datetime(df_pivot['Date']).dt.strftime('%Y-%m-%d')

Pivoting pollution data...


In [5]:
# Load Policy Data and Merge
print("Merging with Policy Data...")
df_policy = pd.read_csv('../data/clean/eu_capitals_covid_data.csv')
df_policy['date'] = pd.to_datetime(df_policy['date']).dt.strftime('%Y-%m-%d')

# Perform a Left Join (Keeps all policy dates, adds pollution data where available)
df_final = pd.merge(
    df_policy, 
    df_pivot, 
    left_on=['city_name', 'date'], 
    right_on=['city_name', 'Date'], 
    how='left'
)

# drop the duplicate 'Date' column from the merge
if 'Date' in df_final.columns:
    df_final = df_final.drop(columns=['Date'])

# save the ultimate combined dataset
output_path = '../data/clean/covid_and_pollution_final.csv'
df_final.to_csv(output_path, index=False)
print(f"Success! Final dataset saved to: {output_path}")

Merging with Policy Data...
Success! Final dataset saved to: ../data/clean/covid_and_pollution_final.csv


In [6]:
# display the result
pd.set_option('display.max_columns', None)
display(df_final.head())

,city_name,date,school_closing,workplace_closing,cancel_events,gatherings_restrictions,transport_closing,stay_home_restrictions,internal_movement_restrictions,international_movement_restrictions,co,dew,humidity,no2,o3,pm10,pm25,precipitation,pressure,so2,temperature,wind gust,wind speed
0,Amsterdam,2020-02-28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.1,NaN,89.2,11.9,21.7,13.0,32.0,NaN,1006.9,0.2,6.6,10.6,4.3
1,Amsterdam,2020-02-29,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.6,NaN,83.3,6.0,20.8,21.0,50.0,NaN,987.1,0.2,9.5,18.9,8.9
2,Amsterdam,2020-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.4,NaN,83.1,5.1,27.4,9.0,19.0,NaN,989.9,0.3,6.1,16.1,6.7
3,Amsterdam,2020-03-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,94.0,13.4,16.4,10.0,25.0,NaN,990.5,0.3,5.0,7.2,3.6
4,Amsterdam,2020-03-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.9,NaN,NaN,11.7,25.2,10.0,23.0,NaN,NaN,0.4,NaN,4.4,2.4
